In [1]:
import time

from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By

In [ ]:
from bs4 import BeautifulSoup
import re
from selenium.webdriver import Keys, ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


options = Options()
options.headless = True
driver = webdriver.Chrome(options=options)
driver.implicitly_wait(5)

driver.get("https://wise.com/")

# Select Source Currency (USD)
source_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.ID, "sourceSelectedCurrency"))
)
source_button.click()

source_input_currency_field = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.ID, "sourceSelectedCurrencySearch"))
)
source_input_currency_field.send_keys("USD")
source_input_currency_field.send_keys(Keys.ENTER)

# Wait for source dropdown to close
# WebDriverWait(driver, 20).until(
#     EC.invisibility_of_element_located((By.ID, "sourceSelectedCurrencySearch"))
# )

# Select Target Currency (BDT)
target_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.ID, "targetSelectedCurrency"))
)
target_button.click()

target_input_currency_field = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.ID, "targetSelectedCurrencySearch"))
)
target_input_currency_field.send_keys("BDT")
target_input_currency_field.send_keys(Keys.ENTER)

# Input Source Amount (1000)
source_input_field_amount = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "input#source"))
)
actions = ActionChains(driver)
actions.click(source_input_field_amount).key_down(Keys.CONTROL).send_keys("a").key_up(Keys.CONTROL).send_keys(Keys.DELETE).perform()
source_input_field_amount.send_keys("1000")
source_input_field_amount.send_keys(Keys.ENTER)

# Wait for UI update
exchange_rate_button = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, "button[aria-describedby='rateLabel']"))
)
exchange_rate = exchange_rate_button.text.strip()

# Extract Transaction Fees
exchange_rate_button = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, "button[aria-describedby='rateLabel']"))
)
exchange_rate = exchange_rate_button.text.strip()  # e.g., "1 USD = 121.842 BDT"

time.sleep(5)

# Extract Transaction Fees with BeautifulSoup
print("Locating fees container...")
fees_container = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, ".Fees_container"))
)
fees_html = fees_container.get_attribute("outerHTML")

# Parse with BeautifulSoup
soup = BeautifulSoup(fees_html, "html.parser")
individual_fees = []

# Extract individual fees from <ul><li>
fee_items = soup.select("ul li.Fees_row")
for item in fee_items:
    fee_name = item.select_one("div:first-child span").text.strip()  # e.g., "Wire transfer fee"
    fee_amount = item.select_one("div:last-child span").text.strip()  # e.g., "6.11 USD"
    individual_fees.append(f"{fee_name}: {fee_amount}")

# Extract total fees
total_fees_row = soup.select_one("div.Fees_row:last-child")
percentage_fee = total_fees_row.select_one("strong:first-child span").text.strip()  # e.g., "Total included fees (1.52%)"
percentage_value = re.search(r'(\d+\.\d+%?)', percentage_fee).group(1) if re.search(r'(\d+\.\d+%?)', percentage_fee) else "Unknown"
flat_fee = total_fees_row.select_one("strong:last-child span").text.strip()  # e.g., "15.17 USD"

# Combine all fees
transaction_fees = f"{', '.join(individual_fees)}, Total: {flat_fee}, Percentage: {percentage_value}"

# Wait for transfer time section to stabilize
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, ".np-section.m-t-2"))
)
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, ".np-section.m-t-2 p.m-b-0 strong"))
)

# Extract Transfer Time with Selenium
transfer_time_container = driver.find_element(By.CSS_SELECTOR, ".tapestry-card-content .np-section.m-t-2 span[role='status']")
transfer_time_html = transfer_time_container.get_attribute("outerHTML")

# Parse transfer time with BeautifulSoup
soup = BeautifulSoup(transfer_time_html, "html.parser")
transfer_time = soup.select_one("p.m-b-0 strong").text.strip()  # e.g., "by Monday"
# Remove "by " prefix if present
transfer_time = transfer_time.replace("by ", "")



print(exchange_rate)
print(transaction_fees)
print(transfer_time)

driver.quit()